In [ ]:
from typing import TypedDict


class State(TypedDict):
    query: str
    answer: str


import time

from langchain.tools import tool
from langgraph.config import get_stream_writer


@tool
def query_database_tool(query: str) -> str:
    """查询数据库工具

    Args:
        query (str): 查询语句

    Returns:
        str: 查询结果
    """
    writer = get_stream_writer()
    for i in range(1, 100):
        time.sleep(0.1)

        writer({"data": f"已检索 {i}/100条记录", "type": "progress"})
    return f"查询 {query} 完毕"


def process_query(state: State) -> State:
    writer = get_stream_writer()
    writer({"status": "正在处理查询请求...", "type": "process"})
    proccessed_query = state["query"].strip()
    if not proccessed_query:
        proccessed_query = "默认查询"
    writer({"status": f"查询 {proccessed_query} 已处理", "type": "info"})
    return {"query": proccessed_query}


def query_database(state: State) -> State:
    result = query_database_tool.invoke(state["query"])
    return {"answer": result}


from langgraph.graph import END, START, StateGraph

graph = (
    StateGraph(State)
    .add_node("process_query", process_query)
    .add_node("query_database", query_database)
    .add_edge(START, "process_query")
    .add_edge("process_query", "query_database")
    .add_edge("query_database", END)
    .compile()
)

async for chunk in graph.astream({"query": "查询明天天气"}, stream_mode="custom"):
    if chunk["type"] == "progress":
        print(chunk, end="\r")
    print(chunk)

{'status': '正在处理查询请求...', 'type': 'process'}
{'status': '查询 查询明天天气 已处理', 'type': 'info'}
{'data': '已检索 1/100条记录', 'type': 'progress'}
{'data': '已检索 2/100条记录', 'type': 'progress'}
{'data': '已检索 3/100条记录', 'type': 'progress'}
{'data': '已检索 4/100条记录', 'type': 'progress'}
{'data': '已检索 5/100条记录', 'type': 'progress'}
{'data': '已检索 6/100条记录', 'type': 'progress'}
{'data': '已检索 7/100条记录', 'type': 'progress'}
{'data': '已检索 8/100条记录', 'type': 'progress'}
{'data': '已检索 9/100条记录', 'type': 'progress'}
{'data': '已检索 10/100条记录', 'type': 'progress'}
{'data': '已检索 11/100条记录', 'type': 'progress'}
{'data': '已检索 12/100条记录', 'type': 'progress'}
{'data': '已检索 13/100条记录', 'type': 'progress'}
{'data': '已检索 14/100条记录', 'type': 'progress'}
{'data': '已检索 15/100条记录', 'type': 'progress'}
{'data': '已检索 16/100条记录', 'type': 'progress'}
{'data': '已检索 17/100条记录', 'type': 'progress'}
{'data': '已检索 18/100条记录', 'type': 'progress'}
{'data': '已检索 19/100条记录', 'type': 'progress'}
{'data': '已检索 20/100条记录', 'type': 'progress'}
